# LensForge: Gravitational Lensing Research Notebook
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-username>/lensforge/blob/main/notebooks/01_point_mass_and_galaxy_lensing.ipynb)

This notebook provides an interactive environment to explore:
1. **Point-Mass Gravitational Lensing**: Einstein radius, thin-screen ray-tracing, analytical image positions, and parity.
2. **Realistic Galaxy Lensing**: Singular Isothermal Sphere (SIS), Singular Isothermal Ellipsoid (SIE, Kormann et al. 1994), External Gravitational Shear, and giant gravitational arcs.


## 1. Environment Setup (Google Colab)
If running in Google Colab, clone the repository and install `lensforge` in editable mode.

In [ ]:
# Check if running in Google Colab
import os, sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("lensforge"):
        !git clone https://github.com/<your-username>/lensforge.git
    %cd lensforge
    !pip install -e .
else:
    print("Running in local environment.")

## 2. Imports & Package Verification

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import lensforge
from lensforge.constants import M_SUN, MEGAPARSEC, arcsec2rad, rad2arcsec
from lensforge.physics.coordinates import make_coordinate_grid
from lensforge.physics.distances import einstein_radius, einstein_radius_arcsec
from lensforge.physics.models import (
    PointMassLens,
    SingularIsothermalSphere,
    SingularIsothermalEllipsoid,
    ExternalShear,
    CompositeLens,
)
from lensforge.imaging.source_models import GaussianSource
from lensforge.visualization.lens_plots import plot_lensing_system, plot_galaxy_lensing_system

print(f"LensForge Version: {lensforge.__version__}")

## 3. Point-Mass Gravitational Lensing
Compute the relativistic Einstein radius for a $10^{11} M_\odot$ lens at $D_l = 1000\text{ Mpc}$ and background source at $D_s = 1800\text{ Mpc}$.

In [ ]:
# 1. Physical cosmological setup
mass_kg = 1.0e11 * M_SUN
d_l_m = 1000.0 * MEGAPARSEC
d_s_m = 1800.0 * MEGAPARSEC
d_ls_m = 1100.0 * MEGAPARSEC

theta_e_rad = einstein_radius(mass_kg, d_l_m, d_s_m, d_ls_m)
theta_e_arcsec = einstein_radius_arcsec(mass_kg, d_l_m, d_s_m, d_ls_m)
print(f"Einstein Radius: {theta_e_rad:.6e} rad = {theta_e_arcsec:.4f} arcsec")

# 2. Instantiate lens model
lens = PointMassLens(theta_e=theta_e_arcsec, center=(0.0, 0.0))

# 3. Background source
src_center = (0.28, 0.18)
source = GaussianSource.from_degrees(
    center=src_center,
    sigma=0.12,
    amp=1.0,
    axis_ratio=0.8,
    pos_angle_deg=35.0,
)

# 4. Image-plane grid & ray-tracing
grid = make_coordinate_grid(num_pix=400, field_of_view=4.0, unit="arcsec")
unlensed = source.evaluate(grid.x, grid.y)
bx, by = lens.ray_trace(grid.x, grid.y, mask_singular=True)
lensed = np.nan_to_num(source.evaluate(bx, by), nan=0.0)

# 5. Analytical image positions for centroid
img_info = lens.image_positions_2d(*src_center)
mag_info = lens.image_magnifications_2d(*src_center)
print(f"Image positions: {img_info['images']}")
print(f"Signed magnifications: mu_+ = {mag_info['mu_plus']:.4f}, mu_- = {mag_info['mu_minus']:.4f}")
print(f"Invariant verification (mu_+ + mu_-): {mag_info['mu_plus'] + mag_info['mu_minus']:.6f} == 1.0")

# 6. Render plot
fig, _ = plot_lensing_system(
    grid=grid,
    unlensed_image=unlensed,
    lensed_image=lensed,
    theta_e=theta_e_arcsec,
    lens_center=(0.0, 0.0),
    source_center=src_center,
    image_positions=img_info['images'],
)
plt.show()

## 4. Realistic Galaxy Lensing: SIE + External Shear
Simulate an elliptical galaxy ($q = 0.72, \phi = 45^\circ$) with tidal external shear ($\gamma = 0.06, \phi_\gamma = 20^\circ$), generating giant gravitational arcs.

In [ ]:
# 1. SIE Halo + External Shear
sie = SingularIsothermalEllipsoid.from_degrees(
    theta_e=1.35,
    axis_ratio=0.72,
    pos_angle_deg=45.0,
    center=(0.0, 0.0),
)
shear = ExternalShear.from_polar_degrees(
    gamma=0.06,
    phi_gamma_deg=20.0,
    center=(0.0, 0.0),
)
composite = CompositeLens([sie, shear])

# 2. Offset background source galaxy
galaxy_src = GaussianSource.from_degrees(
    center=(0.18, 0.12),
    sigma=0.10,
    amp=1.0,
    axis_ratio=0.80,
    pos_angle_deg=30.0,
)

# 3. Ray-tracing and mass convergence
grid_gal = make_coordinate_grid(num_pix=450, field_of_view=4.5, unit="arcsec")
unlensed_gal = galaxy_src.evaluate(grid_gal.x, grid_gal.y)
bx_gal, by_gal = composite.ray_trace(grid_gal.x, grid_gal.y, mask_singular=True)
lensed_gal = np.nan_to_num(galaxy_src.evaluate(bx_gal, by_gal), nan=0.0)
kappa_map = sie.convergence_analytic(grid_gal.x, grid_gal.y, mask_singular=True)

# 4. Render 3-panel visualization
fig, _ = plot_galaxy_lensing_system(
    grid=grid_gal,
    unlensed_image=unlensed_gal,
    lensed_image=lensed_gal,
    convergence_map=kappa_map,
    lens_center=(0.0, 0.0),
    source_center=(0.18, 0.12),
    title="Singular Isothermal Ellipsoid (SIE) + External Shear",
)
plt.show()

## 5. Google Drive Persistence
Mount Google Drive to persist experimental artifacts, simulation grids, or output figures.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/LensForge_Experiments")
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"Experiment artifacts will be saved to: {save_dir}")
else:
    print("Local run: outputs are persisted in ./outputs/")